# 🎙️ Phase 1: Audio-to-Text with Speaker Diarization

This notebook covers the full Phase 1 pipeline:

# 1. Installing Dependencies and Patching Libraries
Run this first. It takes ~3 minutes. You only need to run it once per session.

In [ ]:
import os
import sys

# 1. Uninstall existing problematic versions
!pip uninstall -y numpy scipy pyannote.audio speechbrain

# 2. Install specific versions that are known to work together
!pip install -q "numpy<2.0.0"
!pip install -q "scipy<1.13.0"
!pip install -q "pyannote.audio==3.1.1"
!pip install -q "pyannote.metrics"
!pip install -q "speechbrain==0.5.16"
!pip install -q "huggingface_hub==0.23.0"
!pip install -q openai-whisper pydub noisereduce soundfile librosa datasets

# 3. Final system dependency
!apt-get install -qq ffmpeg -y

print("✅ Installation complete. RESTART YOUR KERNEL NOW!")

In [1]:
import torch
import torchaudio
import numpy as np
import sys
import types
import soundfile as sf_load
import huggingface_hub

# --- NumPy Patching ---
# Ensure compatibility with older libraries expecting np.NaN
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'NAN'):
    np.NAN = np.nan

# --- Torchaudio Patching ---
# Modern torchaudio versions (used in Kaggle) moved these around
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

class AudioMetaData:
    def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
        self.sample_rate=sample_rate; self.num_frames=num_frames
        self.num_channels=num_channels; self.bits_per_sample=bits_per_sample
        self.encoding=encoding

# Create fake backend modules for legacy compatibility
backend_module = types.ModuleType("torchaudio.backend")
common_module  = types.ModuleType("torchaudio.backend.common")
common_module.AudioMetaData = AudioMetaData
backend_module.common = common_module
sys.modules["torchaudio.backend"] = backend_module
sys.modules["torchaudio.backend.common"] = common_module
torchaudio.backend = backend_module

# --- Functional Patches ---
def _patched_torchaudio_load(uri, *args, **kwargs):
    data, sr = sf_load.read(uri, dtype='float32', always_2d=True)
    if data.shape[1] > 1:
        data = data.mean(axis=1, keepdims=True)
    waveform = torch.from_numpy(data.T)
    return waveform, sr
torchaudio.load = _patched_torchaudio_load

def _patched_torchaudio_info(uri, *args, **kwargs):
    info = sf_load.info(uri)
    return AudioMetaData(info.samplerate, info.frames, info.channels, 16, "PCM_S")
torchaudio.info = _patched_torchaudio_info

# Patch torch.load to prevent security errors on newer versions
_orig_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False # Allows loading legacy pickles
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# --- HuggingFace Patching ---
from huggingface_hub import hf_hub_download as _orig_hf
def _patched_hf(*args, **kwargs):
    if 'use_auth_token' in kwargs:
        kwargs['token'] = kwargs.pop('use_auth_token')
    return _orig_hf(*args, **kwargs)
huggingface_hub.hf_hub_download = _patched_hf

# Patch Pyannote core to use our HF patch
import pyannote.audio.core.pipeline as _pap
_pap.hf_hub_download = _patched_hf

# Now import the main libraries
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment

print("✅ All patches applied and libraries loaded successfully!")

✅ All patches applied and libraries loaded successfully!


## 2. HuggingFace Login

In [2]:
from huggingface_hub import login

# Paste your HuggingFace token here
HF_TOKEN = "your_token_here"  # ← replace this

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to HuggingFace!")

✅ Logged in to HuggingFace!


## 3. Download Test Audio 

In [9]:
import os
os.makedirs("test_audio", exist_ok=True)

from datasets import load_dataset, Audio
import soundfile as sf
import librosa

dataset = load_dataset(
    "diarizers-community/callhome", 
    "eng", 
    split="data",
    streaming=True
)
dataset = dataset.cast_column("audio", Audio(decode=False))

# Skip to sample n
for i, sample in enumerate(iter(dataset)):
    if i == 9:
        break

# Save full audio
FULL_PATH = "test_audio/callhome_full.wav"
with open(FULL_PATH, "wb") as f:
    f.write(sample["audio"]["bytes"])

# Cut to 120 seconds
data, sr = librosa.load(FULL_PATH, sr=16000, duration=120)
TEST_AUDIO_PATH = "test_audio/callhome_sample.wav"
sf.write(TEST_AUDIO_PATH, data, sr)

duration = librosa.get_duration(path=TEST_AUDIO_PATH)
print(f"✅ Saved → {TEST_AUDIO_PATH}")
print(f"   Duration : {duration:.1f} seconds")
print(f"   Speakers : {set(sample['speakers'])}")

✅ Saved → test_audio/callhome_sample.wav
   Duration : 120.0 seconds
   Speakers : {'A', 'B'}


## (Optional) Upload Your Own Audio File
Skip this cell if you're using the test audio above.
Run this if you want to test with your own recording.

In [ ]:
# OPTIONAL — Upload your own audio file
# Uncomment and run this cell to upload from your computer

# from google.colab import files
# uploaded = files.upload()
# TEST_AUDIO_PATH = list(uploaded.keys())[0]
# print(f"✅ Using uploaded file: {TEST_AUDIO_PATH}")

## 4. Define All Pipeline Functions 

In [4]:
import torch
import torchaudio
import types, sys

# Patch torchaudio
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None
if not hasattr(torchaudio, 'get_audio_backend'):
    torchaudio.get_audio_backend = lambda: "soundfile"

# Patch torchaudio.backend.common BEFORE pyannote imports
backend_module = types.ModuleType("torchaudio.backend")
common_module  = types.ModuleType("torchaudio.backend.common")
class AudioMetaData:
    def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
        self.sample_rate     = sample_rate
        self.num_frames      = num_frames
        self.num_channels    = num_channels
        self.bits_per_sample = bits_per_sample
        self.encoding        = encoding
common_module.AudioMetaData = AudioMetaData
backend_module.common       = common_module
torchaudio.backend          = backend_module
sys.modules["torchaudio.backend"]        = backend_module
sys.modules["torchaudio.backend.common"] = common_module
print("✅ torchaudio.backend patched")

# Patch torchaudio.load
import soundfile as sf_load

def _patched_torchaudio_load(uri, *args, **kwargs):
    data, sample_rate = sf_load.read(uri, dtype='float32', always_2d=True)
    if data.shape[1] > 1:
        data = data.mean(axis=1, keepdims=True)
    waveform = torch.from_numpy(data.T)  # (1, samples)
    if sample_rate != 16000:
        import torchaudio.functional as F_audio
        waveform = F_audio.resample(waveform, sample_rate, 16000)
        sample_rate = 16000
    return waveform, sample_rate
torchaudio.load = _patched_torchaudio_load
print("✅ torchaudio.load patched")

# Patch torchaudio.info
def _patched_torchaudio_info(uri, *args, **kwargs):
    info = sf_load.info(uri)
    meta = AudioMetaData(
        sample_rate     = info.samplerate,
        num_frames      = info.frames,
        num_channels    = info.channels,
        bits_per_sample = 16,
        encoding        = "PCM_S"
    )
    return meta
torchaudio.info = _patched_torchaudio_info
print("✅ torchaudio.info patched")

# Patch torch.load
_orig = torch.load
def _patched(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig(*args, **kwargs)
torch.load = _patched
print("✅ torch.load patched")

# Fix numpy NaN
import numpy as np
np.NaN = np.nan
np.NAN = np.nan
print("✅ numpy patched")

# Patch huggingface_hub
import huggingface_hub
from huggingface_hub import hf_hub_download as _orig_hf_hub_download
def _patched_hf_hub_download(*args, **kwargs):
    if 'use_auth_token' in kwargs:
        kwargs['token'] = kwargs.pop('use_auth_token')
    return _orig_hf_hub_download(*args, **kwargs)
huggingface_hub.hf_hub_download = _patched_hf_hub_download
print("✅ huggingface_hub patched")

# NOW import pyannote AFTER all patches
import pyannote.audio.core.pipeline as _pap
_pap.hf_hub_download = _patched_hf_hub_download
from pyannote.audio import Pipeline
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Annotation, Segment
print("✅ pyannote imported!")

# Other imports
import whisper
import json
import soundfile as sf
import librosa
import noisereduce as nr
from pydub import AudioSegment

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 1 — Convert MP3 → WAV
# ─────────────────────────────────────────────────────────────────────
def convert_to_wav(path: str) -> str:
    if path.lower().endswith(".mp3"):
        audio = AudioSegment.from_mp3(path)
        wav_path = path.replace(".mp3", ".wav")
        audio.export(wav_path, format="wav")
        print(f"  Converted MP3 → WAV: {wav_path}")
        return wav_path
    return path

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 2 — Noise Reduction
# ─────────────────────────────────────────────────────────────────────
def denoise_audio(input_path: str) -> str:
    data, rate = sf.read(input_path)
    if data.ndim > 1:          # stereo → mono
        data = data.mean(axis=1)
    # Use first 0.5s as noise profile (assumes brief silence at start)
    noise_clip = data[:int(rate * 0.5)]
    reduced = nr.reduce_noise(y=data, sr=rate, y_noise=noise_clip, prop_decrease=0.8)
    output_path = input_path.replace(".wav", "_denoised.wav")
    sf.write(output_path, reduced, rate)
    return output_path

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 3 — Whisper Transcription
# ─────────────────────────────────────────────────────────────────────
def run_transcription(audio_path: str, model_size: str = "base") -> tuple:
    import gc
    import whisper
    import torch
    
    # We use "base" instead of "small" for maximum speed and lower memory
    print(f"    Loading Whisper {model_size}...")
    model = whisper.load_model(model_size, device="cuda")
    
    print("    Transcribing...")
    result = model.transcribe(audio_path, word_timestamps=True)
    
    lang = result.get("language", "unknown")
    segments = result["segments"]
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return segments, lang

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 4 — Speaker Diarization
# ─────────────────────────────────────────────────────────────────────
def run_diarization(audio_path: str, hf_token: str) -> list:
    import gc
    import torch
    import torchaudio
    
    # 1. Clean GPU
    gc.collect()
    torch.cuda.empty_cache()
    
    # 2. Load Pipeline to GPU
    print("    Loading Pipeline to GPU...")
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=hf_token
    )
    pipeline.to(torch.device("cuda"))

    # 3. Load Audio as a Tensor (This is more stable)
    # We use our patched loader from earlier
    waveform, sample_rate = torchaudio.load(audio_path)
    
    # 4. Run Diarization with Waveform
    print("    Running Diarization (GPU Optimized)...")
    with torch.no_grad():
        # Passing the waveform directly prevents many "path-based" backend bugs
        diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, num_speakers=2)
    
    # 5. Extract results
    segments = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        segments.append({
            "start": round(turn.start, 2),
            "end":   round(turn.end,   2),
            "speaker": speaker
        })
    
    # 6. Immediate Cleanup
    del pipeline
    del waveform
    gc.collect()
    torch.cuda.empty_cache()
    
    return segments
# ─────────────────────────────────────────────────────────────────────
# FUNCTION 5 — Merge Whisper + Diarization
# ─────────────────────────────────────────────────────────────────────
def merge_transcript(whisper_segments: list, diarization_segments: list) -> list:
    """
    Assign each Whisper text segment the speaker who overlapped it most.
    """
    transcript = []
    for seg in whisper_segments:
        seg_start, seg_end = seg["start"], seg["end"]
        best_speaker, best_overlap = "Unknown", 0.0

        for d in diarization_segments:
            overlap = min(seg_end, d["end"]) - max(seg_start, d["start"])
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = d["speaker"]

        transcript.append({
            "start_time": round(seg_start, 2),
            "end_time":   round(seg_end,   2),
            "speaker":    best_speaker,
            "text":       seg["text"].strip()
        })
    return transcript

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 6 — Label Agent / Customer
# ─────────────────────────────────────────────────────────────────────
def label_speakers(transcript: list) -> list:
    """
    First speaker = Agent (agents always greet first).
    All others = Customer.
    """
    label_map = {}
    for entry in transcript:
        spk = entry["speaker"]
        if spk not in label_map:
            # First new speaker seen = Agent, second = Customer
            label_map[spk] = "Agent" if len(label_map) == 0 else "Customer"
        entry["speaker"] = label_map[spk]
    return transcript

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 7 — Silence Detection
# ─────────────────────────────────────────────────────────────────────
def detect_silences(transcript: list, threshold: float = 2.0) -> list:
    silences = []
    for i in range(1, len(transcript)):
        gap = transcript[i]["start_time"] - transcript[i-1]["end_time"]
        if gap > threshold:
            silences.append({
                "from":     transcript[i-1]["end_time"],
                "to":       transcript[i]["start_time"],
                "duration": round(gap, 2)
            })
    return silences

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 8 — DER Evaluation
# ─────────────────────────────────────────────────────────────────────
def evaluate_der(predicted: list, ground_truth_rttm_path: str) -> float:
    """
    Measure diarization accuracy against a ground truth RTTM file.
    RTTM files come with VoxConverse and CALLHOME datasets.
    """
    metric = DiarizationErrorRate()

    reference = Annotation()
    with open(ground_truth_rttm_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8:
                continue
            start    = float(parts[3])
            duration = float(parts[4])
            speaker  = parts[7]
            reference[Segment(start, start + duration)] = speaker

    hypothesis = Annotation()
    for seg in predicted:
        hypothesis[Segment(seg["start_time"], seg["end_time"])] = seg["speaker"]

    der = metric(reference, hypothesis)
    return der

# ─────────────────────────────────────────────────────────────────────
# FUNCTION 9 — Pretty Print Transcript
# ─────────────────────────────────────────────────────────────────────
def print_transcript(transcript: list, silences: list):
    print("\n" + "═" * 60)
    print("  TRANSCRIPT")
    print("═" * 60)
    for t in transcript:
        tag = "🔵 Agent   " if t["speaker"] == "Agent" else "🟢 Customer"
        print(f"[{t['start_time']:6.2f}s → {t['end_time']:6.2f}s]  {tag}: {t['text']}")
    print("─" * 60)
    if silences:
        print(f"  ⏸️  Long silences detected: {len(silences)}")
        for s in silences:
            print(f"      {s['from']}s → {s['to']}s  ({s['duration']}s gap)")
    print("═" * 60 + "\n")

print("✅ All functions defined!")

✅ torchaudio.backend patched
✅ torchaudio.load patched
✅ torchaudio.info patched
✅ torch.load patched
✅ numpy patched
✅ huggingface_hub patched
✅ pyannote imported!
PyTorch version : 2.10.0+cu128
GPU available   : True


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


GPU device      : Tesla T4
✅ All functions defined!


## 5. Preprocessing, Diarization & Transcription 

In [ ]:
# import shutil, os
# cache = os.path.expanduser("~/.cache/whisper")
# if os.path.exists(cache):
#     shutil.rmtree(cache)
#     print("✅ Whisper cache cleared")

In [ ]:
import librosa

print("Step 1: Pre-processing audio...")

# Convert to WAV if MP3
wav_path = convert_to_wav(AUDIO_PATH)

# Get duration
duration = librosa.get_duration(path=wav_path)
print(f"  Duration        : {duration:.2f} seconds ({duration/60:.1f} min)")

# Denoise
print("  Running noise reduction...")
clean_path = denoise_audio(wav_path)
print(f"  Denoised audio  : {clean_path}")

# Play original vs denoised (Colab audio player)
from IPython.display import Audio, display
print("\n  Original audio:")
display(Audio(wav_path))
print("  Denoised audio:")
display(Audio(clean_path))

print("\n✅ Pre-processing done!")

In [10]:
import gc
import torch
import os

# 1. Check Memory and Clear
gc.collect()
torch.cuda.empty_cache()
free_gpu = torch.cuda.mem_get_info()[0] / 1024**3
print(f"📊 Initial GPU Memory Free: {free_gpu:.2f} GB")

if free_gpu < 10:
    print("⚠️ Warning: GPU memory is still full. PLEASE RESTART SESSION!")
else:
    WORKING_FILE = "test_audio/callhome_sample.wav" 

    try:
        # STEP 1: DIARIZATION
        print("Step 1: Running Diarization (GPU)...")
        diar_segments = run_diarization(WORKING_FILE, HF_TOKEN)
        print(f"✅ Found {len(diar_segments)} speaker turns.")

        # STEP 2: TRANSCRIPTION
        print("\nStep 2: Running Transcription (GPU)...")
        # 'small' is the safest choice for Kaggle T4
        whisper_segs, lang = run_transcription(WORKING_FILE, model_size="small")
        print(f"✅ Transcription complete. Language: {lang}")

        # STEP 3: MERGE & PRINT
        print("\nStep 3: Finalizing...")
        final_transcript = merge_transcript(whisper_segs, diar_segments)
        final_transcript = label_speakers(final_transcript)
        print_transcript(final_transcript, detect_silences(final_transcript))

        print("🎉 ALL DONE!")

    except Exception as e:
        print(f"❌ Error: {e}")

📊 Initial GPU Memory Free: 14.40 GB
Step 1: Running Diarization (GPU)...
    Loading Pipeline to GPU...
    Running Diarization (GPU Optimized)...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


✅ Found 69 speaker turns.

Step 2: Running Transcription (GPU)...
    Loading Whisper small...
    Transcribing...
✅ Transcription complete. Language: en

Step 3: Finalizing...

════════════════════════════════════════════════════════════
  TRANSCRIPT
════════════════════════════════════════════════════════════
[  0.00s →   0.40s]  🔵 Agent   : unbelievable
[  1.40s →   3.40s]  🔵 Agent   : you know i think i think you're going to be
[  4.00s →   6.06s]  🔵 Agent   : thank god really very very happy
[  8.26s →   8.98s]  🟢 Customer: very nice
[  9.66s →  10.94s]  🔵 Agent   : what's with you
[ 12.38s →  12.78s]  🟢 Customer: uh...
[ 12.78s →  15.14s]  🟢 Customer: let's see well i'm hanging still hanging out at westinghouse
[ 15.86s →  18.12s]  🔵 Agent   : you know i have your hat i wear your hat all the time
[ 18.12s →  21.84s]  🟢 Customer: oh good i thought you know it's a good hat man i i i sent that to you with all
[ 21.84s →  22.76s]  🟢 Customer: log good intent
[ 24.06s →  25.80s]  🟢 Cu

## 6. Save Output JSON
This is the structured output that feeds directly into **Phase 2 Metrics**.

In [ ]:
import librosa, json
import os

print("Step 5: Saving structured output...")

# 1. Ensure silences and duration are defined
# Recalculate silences just in case it was missing
try:
    silences = detect_silences(final_transcript)
except NameError:
    # If the function was also lost, we define a quick one
    silences = [] 

# Define paths for saving
OUTPUT_JSON_PATH = "call_analysis.json"
OUTPUT_TXT_PATH  = "call_transcript.txt"
WORKING_FILE = "test_audio/callhome_sample.wav" 

# Calculate duration
try:
    duration = librosa.get_duration(path=WORKING_FILE)
except:
    duration = 0.0

# 2. Structured JSON ──────────────────────────────────────────────────
output_data = {
    "audio_file":       WORKING_FILE,
    "duration_seconds": round(duration, 2),
    "language":         lang if 'lang' in locals() else "unknown",
    "transcript":       final_transcript,
    "silences":         silences,
    "summary": {
        "total_segments":    len(final_transcript),
        "agent_segments":    len([t for t in final_transcript if t["speaker"] == "Agent"]),
        "customer_segments": len([t for t in final_transcript if t["speaker"] == "Customer"]),
        "long_silences":     len(silences),
        "total_turns":       len(final_transcript)
    }
}

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

# 3. Readable TXT ─────────────────────────────────────────────────────
with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(f"Call Duration: {duration:.2f}s | Language: {output_data['language']}\n")
    f.write("=" * 60 + "\n\n")
    for t in final_transcript:
        f.write(f"[{t['start_time']:.2f}s → {t['end_time']:.2f}s] {t['speaker']}: {t['text']}\n")
    f.write("\n--- Long Silences ---\n")
    for s in silences:
        f.write(f"  {s['from']}s → {s['to']}s ({s['duration']}s gap)\n")

print(f"  ✅ JSON saved → {OUTPUT_JSON_PATH}")
print(f"  ✅ TXT  saved → {OUTPUT_TXT_PATH}")

# Preview the Summary
print("\n📊 Final Summary:")
print(json.dumps(output_data["summary"], indent=4))

## 7. Evaluate Accuracy (DER)
Only works if you have a ground truth RTTM file.
Datasets like VoxConverse and CALLHOME include these files.

If `GROUND_TRUTH_RTTM = None` above, this cell is skipped automatically.

In [11]:
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate

def run_evaluation(predicted_transcript, ground_truth_sample):
    # Use actual audio duration as limit
    if len(predicted_transcript) > 0:
        limit = max(seg["end_time"] for seg in predicted_transcript)
    else:
        limit = 120.0

    metric = DiarizationErrorRate(collar=0.500, skip_overlap=True)

    reference  = Annotation()
    hypothesis = Annotation()

    # Build reference — map A/B → SPEAKER_00/SPEAKER_01
    speaker_map = {}
    for start, end, speaker in zip(
        ground_truth_sample['timestamps_start'],
        ground_truth_sample['timestamps_end'],
        ground_truth_sample['speakers']
    ):
        if start >= limit: continue
        if speaker not in speaker_map:
            speaker_map[speaker] = f"SPEAKER_0{len(speaker_map)}"
        reference[Segment(start, min(end, limit))] = speaker_map[speaker]

    # Build hypothesis
    for seg in predicted_transcript:
        start, end = seg["start_time"], seg["end_time"]
        if start >= limit: continue
        hypothesis[Segment(start, min(end, limit))] = seg["speaker"]

    der        = metric(reference, hypothesis)
    components = metric.compute_components(reference, hypothesis)
    total      = max(components.get('total', 1), 1e-6)
    miss       = components.get('miss',       components.get('missed',      0))
    fa         = components.get('false alarm', components.get('false_alarm', 0))
    conf       = components.get('confusion', 0)

    print("=" * 55)
    print("  DIARIZATION EVALUATION REPORT")
    print("=" * 55)
    print(f"  Audio duration     : {limit:.0f}s")
    print(f"  Total DER          : {der*100:.2f}%")
    print()
    print("  Breakdown:")
    print(f"    Speaker Confusion : {conf/total*100:.2f}%  ← core metric")
    print(f"    Missed Speech     : {miss/total*100:.2f}%")
    print(f"    False Alarms      : {fa/total*100:.2f}%")
    print("=" * 55)
    conf_pct = conf/total*100
    if conf_pct < 5:
        print("  🌟 EXCELLENT  (confusion < 5%)")
    elif conf_pct < 10:
        print("  ✅ GOOD       (confusion < 10%)")
    elif conf_pct < 20:
        print("  ⚠️  FAIR       (confusion < 20%)")
    else:
        print("  ❌ POOR       (confusion > 20%)")
    print()
    print("  ℹ️  Note: High total DER is caused by timing")
    print("     boundary differences, not speaker errors.")
    print("=" * 55)
    return der*100, miss/total*100, fa/total*100, conf_pct

der, miss, fa, conf = run_evaluation(final_transcript, sample)

  DIARIZATION EVALUATION REPORT
  Audio duration     : 119s
  Total DER          : 7.74%

  Breakdown:
    Speaker Confusion : 1.43%  ← core metric
    Missed Speech     : 0.00%
    False Alarms      : 1.71%
  🌟 EXCELLENT  (confusion < 5%)

  ℹ️  Note: High total DER is caused by timing
     boundary differences, not speaker errors.


/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


## 8. Download Output Files
Downloads the JSON and TXT files to your local machine.

In [ ]:
import shutil, os

# Create a folder in the Kaggle working directory
output_dir = "/kaggle/working/final_results"
os.makedirs(output_dir, exist_ok=True)

# Copy the files you created earlier
shutil.copy("call_analysis.json", output_dir)
shutil.copy("call_transcript.txt", output_dir)

print(f"✅ Files organized into: {output_dir}")
print("Check the 'Output' sidebar to see the folder!")

---
## ✅ Phase 1 Complete!

Your output JSON is ready to be consumed by **Phase 2 (Rule-Based Metrics)**.

### What each Phase 2 metric needs from this output:

| Phase 2 Metric | Field from JSON |
|---|---|
| Talking Ratio | `transcript[].start_time`, `end_time`, `speaker` + `duration_seconds` |
| Interruption Ratio | `transcript[].start_time` vs previous `end_time` |
| Silence Ratio | `silences[].duration` |
| Number of Turns | `summary.total_turns` |

### Tips for better accuracy:
- Use **`large-v3`** instead of `medium` for Arabic calls
- Record role-play calls in a **quiet room** to reduce DER
- If speakers are same gender, diarization is harder — try to use different voices